# Semana 3 · Sesión 2: GitHub y notebooks bajo control de versiones

**Módulo 0**

## Objetivos de la sesión

1. Configurar tu fork del curso con `origin` y `upstream`, y mantenerlo
   sincronizado.
2. Abrir un Pull Request dentro de tu propio fork y usar Issues para pedir
   ayuda.
3. Explicar por qué el diff de un `.ipynb` es ilegible y cómo lo arregla
   `nbstripout`.


## Antes de empezar

En la sesión 1 usamos Git entero dentro de tu máquina. Hoy sale a internet:
copias del repositorio en GitHub, y el flujo con el que vas a entregar
todas las tareas del resto del semestre.

Este notebook no necesita el repositorio de práctica de la sesión 1 — se
puede correr por su cuenta. Vuelve a haber terminal —los bloques de comandos se
teclean ahí— y hoy sí hay Python: la segunda mitad de la sesión es sobre el
archivo `.ipynb` visto por dentro.

Ten tu sesión de GitHub abierta en el navegador: al final de la clase vas a
salir con tu fork configurado y tu primer Pull Request abierto.


In [ ]:
import copy
import difflib
import json

## Remotos: el mismo repositorio, en otra máquina

Un **remoto** es una copia de tu repositorio que vive en otro lado —en
GitHub, normalmente— y a la que le pones un nombre corto. El nombre por
convención de "el mío en GitHub" es `origin`.

```bash
git clone <url>        # trae un repositorio completo, con toda su historia
git remote -v          # ¿qué remotos conozco y con qué URL?
git push origin main   # manda mis commits de main al remoto
git pull origin main   # trae los commits que haya allá y los fusiona
```

`git clone` no es un comando aparte de lo que ya sabes: hace un `init`,
configura `origin` y trae la historia completa. Después de clonar, todo lo
de la sesión 1 funciona igual.

Fíjate en que `push` y `pull` son los únicos comandos de la sesión de hoy
que necesitan internet. Commitear, ramificar y fusionar siguen siendo
locales.


## Fork: tu copia del curso

Un **fork** es una copia completa de un repositorio ajeno, en tu cuenta de
GitHub. Es tu repositorio personal del curso: ahí trabajas, ahí entregas, y
de ahí traes el material nuevo cada semana.

Vas a tener dos remotos configurados, y conviene no confundirlos nunca:

| Remoto | Apunta a | Puedes escribir | Para qué |
|---|---|---|---|
| `origin` | Tu fork | Sí | Subir tu trabajo y tus entregas |
| `upstream` | El repositorio del curso | No | Traer el material nuevo de cada semana |

El ciclo de cada semana es siempre el mismo:

```bash
git fetch upstream        # ¿qué hay de nuevo en el curso?
git merge upstream/main   # tráelo a mi rama actual
git push origin main      # y actualiza también mi copia en GitHub
```

Todo esto está escrito paso a paso, con el detalle de cada botón, en
[`docs/git-guia.md`](../../docs/git-guia.md). Esa guía es la referencia que
vas a consultar el resto del semestre; hoy la recorremos juntos una vez.


## TODO en clase 1

Configura tu fork. Es la primera mitad del entregable de la semana:

1. En la página del repositorio del curso, da clic en **Fork** y confirma.
2. En tu fork recién creado, copia la URL del botón verde **Code**.
3. En tu terminal:

```bash
git clone <la URL de TU fork>
cd temas-selectos-fisica-computacional-1

git ____ -v                        # ¿qué remotos tengo ahora?

git remote add ____ https://github.com/moiseszeleny/temas-selectos-fisica-computacional-1.git

git ____ -v                        # ahora deben aparecer dos: origin y upstream
```

Comprueba antes de seguir: `origin` debe tener **tu** usuario en la URL, y
`upstream` el del profesor. Si están al revés, bórralo con
`git remote remove <nombre>` y vuelve a agregarlo.


## Pull Requests

Un **Pull Request** (PR) es una propuesta de fusión: "quiero traer los
commits de esta rama a esta otra, revísenlos". Es el mecanismo con el que
se revisa código en todo el mundo, y en este curso es la forma de entregar
tareas a partir de hoy.

Con una diferencia que hay que subrayar, porque es donde más gente se
equivoca: **tu PR va dentro de tu propio fork**, no contra el repositorio
del curso. Base y comparación son ramas tuyas.

Al abrir el PR, GitHub muestra arriba dos selectores. El primero,
**base repository**, viene por defecto con el repositorio del curso:
cámbialo a `<tu-usuario>/temas-selectos-fisica-computacional-1`. Si tu PR
aparece con decenas de commits que no escribiste, es exactamente este el
error.

Lo que pasa cuando lo abres:

- GitHub Actions corre los tests de la tarea y deja un check ✅ o ❌ en el
  PR. No tienes que instalar nada para que eso ocurra.
- El asistente lee tu código y deja comentarios línea por línea ahí mismo.


## Issues: dónde preguntar

Un **Issue** es una conversación con título, abierta en un repositorio,
para reportar un problema o hacer una pregunta. Si algo no te funciona —el
entorno, un test que no entiendes, un conflicto que se resiste— abre un
Issue en tu fork: queda escrito, se puede responder con código y calma, y
casi siempre le sirve a alguien más.

Es mejor que un mensaje suelto justamente porque deja rastro. Y saber
escribir uno bueno (qué esperabas, qué pasó, qué comando corriste, qué
error salió) es una habilidad que vale más allá de este curso.


## TODO en clase 2

Tu primer Pull Request. Es la segunda mitad del entregable de la semana:
agregarte a `docs/roster.md`, para que el asistente sepa dónde encontrar
tus entregas.

```bash
git switch -c ____                 # una rama, por ejemplo "agrega-mi-fila-al-roster"

# edita docs/roster.md y agrega TU fila al final de la tabla:
# | tu-usuario-de-github | https://github.com/tu-usuario/temas-selectos-fisica-computacional-1 |

git ____ docs/roster.md
git ____ -m "____"                 # mensaje en imperativo
git ____ origin <tu rama>          # sube la rama a TU fork
```

Y en GitHub, dentro de **tu fork**: pestaña **Pull requests** → **New pull
request** → revisa que **base repository** sea el tuyo → base `main`,
compare tu rama → **Create pull request**.

Revísalo antes de darle crear: debe mostrar **un solo commit** y **un solo
archivo cambiado**. Si muestra más, casi seguro el base repository quedó
apuntando al repositorio del curso.


## Un notebook es un archivo JSON

Cambiamos de tema, a la parte que hace que Git y los notebooks se lleven
mal.

Un `.ipynb` no es un formato misterioso: es un archivo de texto en formato
**JSON**, un diccionario con una lista de celdas. Vamos a abrir uno con
Python —igual que lo abre Jupyter— para ver qué hay dentro.

El archivo de ejemplo es un notebook corto de caída libre, guardado
**con sus salidas**: tres celdas de código, un número impreso y una
gráfica.


In [ ]:
with open("ejemplos/notebook-con-salidas.json", encoding="utf-8") as archivo:
    notebook = json.load(archivo)

list(notebook), len(notebook["cells"])

Un aparte que viene al caso: ese archivo termina en `.json` y no en
`.ipynb` a propósito. Este repositorio tiene configurado un filtro que
**borra las salidas de todo archivo `.ipynb` al hacer commit** —lo veremos
en un momento—, así que un ejemplo con salidas guardado como `.ipynb` se
habría limpiado solo, y nos habríamos quedado sin ejemplo.

Cada celda es un diccionario. Las llaves que importan hoy:

| Llave | Qué guarda |
|---|---|
| `cell_type` | `"code"` o `"markdown"` |
| `source` | El texto de la celda, como lista de líneas |
| `outputs` | Las salidas de la última ejecución (solo celdas de código) |
| `execution_count` | El número entre corchetes, `In [3]` |


In [ ]:
celda = notebook["cells"][3]

print("tipo:", celda["cell_type"], "· execution_count:", celda["execution_count"])
print("fuente:", repr("".join(celda["source"])[:40]), "...")
print("salidas:", [salida["output_type"] for salida in celda["outputs"]])

imagen = celda["outputs"][1]["data"]["image/png"]
print("la gráfica ocupa", len(imagen), "caracteres de base64")
print("empieza así:", imagen[:48], "...")

## Por qué el diff de un notebook es ilegible

Ahora el experimento. Supongamos que corriges **una línea** del notebook —la
altura inicial, de 100 m a 120 m— y lo vuelves a ejecutar. Para ti cambió
una celda; veamos qué ve Git.

Al reejecutar cambian tres cosas: la línea de código, el número impreso y
la gráfica, que es una cadena de base64 completamente distinta.


In [ ]:
notebook_v2 = copy.deepcopy(notebook)

# 1. el cambio de verdad: una línea de código
fuente = "".join(notebook_v2["cells"][2]["source"]).replace("100 -", "120 -")
notebook_v2["cells"][2]["source"] = fuente.splitlines(keepends=True)

# 2. al reejecutar, el número impreso cambia...
notebook_v2["cells"][2]["outputs"][0]["text"] = ["Altura a los 3 s: 75.85 m\n"]
notebook_v2["cells"][2]["execution_count"] = 4

# 3. ...y la gráfica se vuelve otra imagen: otra cadena de base64, del
# mismo tamaño pero distinta de principio a fin
imagen_original = notebook["cells"][3]["outputs"][1]["data"]["image/png"]
notebook_v2["cells"][3]["outputs"][1]["data"]["image/png"] = imagen_original[::-1]
notebook_v2["cells"][3]["execution_count"] = 5

In [ ]:
lineas_v1 = json.dumps(notebook, indent=1).splitlines(keepends=True)
lineas_v2 = json.dumps(notebook_v2, indent=1).splitlines(keepends=True)

diferencias = list(difflib.unified_diff(lineas_v1, lineas_v2))
cambiadas = [
    linea
    for linea in diferencias
    if linea[0] in "+-" and not linea.startswith(("+++", "---"))
]

print("celdas que cambiaron de verdad:", 1)
print("líneas que Git marca como distintas:", len(cambiadas))
print("caracteres en esas líneas:", sum(len(linea) for linea in cambiadas))

Ese es el problema completo, en tres números. Cambiaste una celda, y el
diff que le queda a quien revise tu trabajo son miles de caracteres, casi
todos base64 de una imagen. Las consecuencias diarias:

- **El diff no se puede leer**: `git diff` deja de servir para contestar
  "¿qué cambió?".
- **Los conflictos se vuelven irresolubles a mano**: si dos personas
  ejecutan el mismo notebook, chocan en la línea de la imagen — y esa línea
  no se puede editar a ojo para "quedarse con la versión correcta".
- **El repositorio se infla**: cada ejecución guarda una copia nueva de
  cada gráfica, para siempre.

Y fíjate en lo que *sí* cambió de verdad: una línea de código. Todo lo
demás es ruido de ejecución.


## `nbstripout`: quitar las salidas antes de guardar

La solución es no versionar las salidas. Un notebook sin salidas conserva
todo lo que escribiste —código y markdown— y pierde solo lo que se puede
regenerar corriéndolo.

`nbstripout` hace exactamente eso, y se engancha a Git como un **filtro**:
se ejecuta solo, cada vez que preparas un `.ipynb` para un commit. En tu
copia del archivo las salidas siguen ahí; lo que se guarda en el historial
va limpio.

En este repositorio ya está configurado. El archivo `.gitattributes` de la
raíz dice qué archivos pasan por el filtro:

```
*.ipynb filter=nbstripout
*.ipynb diff=ipynb
```

Y cada quien lo activa una sola vez por clon —hazlo hoy en tu fork, si no
lo has hecho:

```bash
nbstripout --install --attributes .gitattributes
```

Existe otra estrategia, **jupytext**, que mantiene el notebook emparejado
con un archivo `.py` legible y versiona ese; es útil cuando el código pesa
más que las gráficas. No la vamos a usar en el curso, pero vale la pena que
sepan que existe.


## TODO en clase 3

Pongámosle número a la afirmación "casi todo el archivo son salidas".
Completa la función y córrela sobre el notebook de ejemplo:

```python
len(json.dumps(notebook))   # cuántos caracteres pesa el archivo completo
```

La fracción que buscamos es: caracteres de salidas ÷ caracteres totales.


In [ ]:
# TODO en clase: suma cuántos caracteres ocupan las salidas de todo el notebook
def caracteres_de_salidas(notebook):
    total = 0
    for celda in notebook["cells"]:
        if celda["cell_type"] == "code":
            total += ...   # ¿cuánto pesa json.dumps de las salidas de esta celda?
    return total


# Al terminar, quita el comentario de estas dos líneas y córrelas:
# fraccion = caracteres_de_salidas(notebook) / len(json.dumps(notebook))
# print(f"{fraccion:.1%} del archivo son salidas")

## Así se desarrolla SymPy

Nada de lo de hoy es exclusivo de este curso. SymPy —la biblioteca con la
que trabajaremos a partir de la próxima semana— se desarrolla exactamente
así: su código vive en [github.com/sympy/sympy](https://github.com/sympy/sympy),
cada cambio entra por un Pull Request, cada PR pasa por tests automáticos y
por la revisión de otras personas, y las discusiones quedan en los Issues.

Ábranlo un momento y miren la pestaña **Pull requests**. Es el mismo flujo
que acaban de usar para agregarse al roster, con más gente y más tests.

Su proyecto final del semestre va a vivir así también: en un repositorio,
con su historia, sus ramas y sus PRs.


## Resumen

Hoy sacamos Git a la red: remotos, fork con `origin` y `upstream`, el ciclo
semanal de sincronización, y el Pull Request como forma de entregar y de
recibir revisión. Después vimos el `.ipynb` por dentro —un JSON con celdas
y salidas—, por qué su diff es ilegible, y cómo `nbstripout` lo arregla
quitando las salidas antes de que lleguen al historial.

**Tarea de esta semana:** son **dos entregas**.

1. Tu **Pull Request al roster**, dentro de tu propio fork (el TODO 2 de
   hoy). Si te faltó terminarlo en clase, la guía completa está en
   [`docs/git-guia.md`](../../docs/git-guia.md).
2. [`tarea-03.ipynb`](../tarea/tarea-03.ipynb) — entrega antes de la clase
   de la Semana 4, también por Pull Request dentro de tu fork.

**Próxima clase — Semana 4:** expresiones simbólicas. Empezamos SymPy de
verdad: símbolos, suposiciones y manipulación de expresiones.
